## Predict Delayed Deliveries Agent — OpenAI GPT-5.4

```perl

┌─────────────────────────────────────────────────────────────┐
│  User submits query via Gradio                              │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  delivery_chat_app.py                                       │
│  1. Append file path to query (if uploaded)                 │
│  2. Check sidecar mtime → append [SYSTEM: FRESH/NOT FRESH]  │
│  3. Runner.run_streamed(master_agent, full_query)           │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Master Agent (master_expert.md)                            │
│  Sees "predict" request → calls predict_delivery_delays_tool│
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Predict Agent (predict_delivery_delays.md)                 │
│  Calls predict_delivery_delays MCP tool (exactly once)      │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  prediction_server.py (MCP stdio)                           │
│  → DailyPredictionPipeline.get_prediction()                 │
│    ├── Stage 1: Delay RF (delayed vs on-time)               │
│    ├── Stage 2: Severity RF (Short/Medium/Long)             │
│    ├── Write predictions to SQLite summary tables           │
│    ├── Save delayed CSV → data/processed/                   │
│    ├── Save sidecar JSON (summary + formatted_stats) → disk │
│    └── Return JSON: {summary, formatted_stats,              │
│                      delayed_orders[0:10]}                  │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Predict Agent processes tool output                        │
│  ├── IGNORES summary + formatted_stats (not in schema)      │
│  ├── Fills llm_insights for each of 10 rows                 │
│  │   (cross-functional analysis referencing derived features)│
│  └── Writes predict_summary (Markdown bullets)              │
│                                                             │
│  OUTPUT: {predict_summary, delayed_orders: [{id, insights}]}│
│  (only 2 fields — entire output budget for intelligence)    │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Master Agent receives predict tool result                  │
│  ├── predict_summary → MasterOutput.predict_summary         │
│  └── delayed_orders  → MasterOutput.predict_rows            │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  delivery_app.py processes MasterOutput                     │
│  ├── Guard: isinstance check (skip if str from weak model)  │
│  ├── Heading guard: prepend ### if missing                  │
│  ├── Read sidecar JSON from disk → formatted_stats + csv_path│
│  ├── Append formatted_stats to predict_text                 │
│  ├── Read CSV from disk → predict_df                        │
│  ├── Merge llm_insights by delivery_id into predict_df      │
│  ├── Save updated CSV                                       │
│  └── Display in Gradio (text + table + download)            │
└─────────────────────────────────────────────────────────────┘
```


### Predict Delay Fixes

| # | Fix | File | Why |
|---|---|---|---|
| 1 | Rounding `vehicle_load_strain`, `km_per_expected_hr` to 2dp | `daily_predict.py` | Raw floats had 10+ decimals, wasting tokens and confusing the LLM |
| 2 | Row cap restored to 10 | `daily_predict.py` | Limit cost/tokens during testing; controlled by `SC_MCP_ENRICH_ROWS` env var |
| 3 | Tracing disabled | `.env` | `OPENAI_AGENTS_DISABLE_TRACING=1` — 10KB trace payload limit was blocking requests |
| 4 | Schema slimmed from 15 fields to 2 per row | `delivery_agents.py` | `RowEnrichment(delivery_id, llm_insights)` — LLM doesn't copy CSV data it doesn't need |
| 5 | `llm_insights` made required with `min_length=10` | `delivery_agents.py` | Was `Optional[str]=None` — LLM would skip it silently |
| 6 | Sidecar JSON file (the key architectural fix) | `daily_predict.py`, `delivery_app.py` | Pipeline saves `summary + formatted_stats` to disk; app reads from disk. LLM never touches pass-through data |
| 7 | Removed pass-through fields from agent output models | `delivery_agents.py` | `DeliveryDelayPredictionResult` → only `predict_summary + delayed_orders`. `MasterOutput` stripped of `predict_formatted_stats + predict_csv_path` |
| 8 | Prompt rewrite — "YOUR OUTPUT HAS ONLY 2 FIELDS" | `predict_delivery_delays.md`, `master_expert.md` | Clear instructions, few-shot examples for `llm_insights`, Step 1/2 for `predict_summary` |
| 9 | Heading guard | `delivery_app.py` | Prepends `### Cross-Dimensional Delay Insights` if model omits the heading |
| 10 | String fallback guard | `delivery_app.py` | Ensures output remains valid if the model returns a plain string instead of structured output |

---

## Diagnosys of Delay Patterns

```perl
┌─────────────────────────────────────────────────────────────┐
│  User submits "Provide delay pattern diagnosis"             │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  delivery_app.py                                            │
│  Check sidecar file mtime:                                  │
│  ├── exists && < 1h old → [SYSTEM: FRESH]                   │
│  └── missing or > 1h   → [SYSTEM: NOT FRESH]                │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Master Agent reads system note                             │
│                                                             │
│  ┌─ FRESH ──────────────────────────────────────────┐       │
│  │  Skip predict, call diagnose directly            │       │
│  └──────────────────────────────────────────────────┘       │
│                                                             │
│  ┌─ NOT FRESH ──────────────────────────────────────┐       │
│  │  1. Call predict_delivery_delays_tool (prereq)   │       │
│  │  2. Wait for output                              │       │
│  │  3. Then call diagnose_delay_patterns_tool       │       │
│  └──────────────────────────────────────────────────┘       │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Diagnosis Agent (diagnose_delay_patterns.md)               │
│  Calls get_delay_diagnosis MCP tool (no arguments)          │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  prediction_server.py (MCP stdio)                           │
│  → DatabaseOperations.get_diagnosis_data()                  │
│    ├── Reads daily_summary_* tables (written by predict)    │
│    ├── Reads hist_summary_* tables (historical baseline)    │
│    └── Returns: {overall_daily, overall_hist, comparison,   │
│                  daily_high_risk_patterns,                  │
│                  hist_high_risk_patterns}                   │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Diagnosis Agent processes tool output                      │
│  ├── Copies high_risk_patterns + comparison                 │
│  └── Writes diagnosis_summary (Markdown: overall, worsening,│
│      improving, high-risk combos, root cause analysis)      │
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  Master Agent receives diagnosis tool result                │
│  ├── diagnosis_summary     → MasterOutput.diagnosis_summary │
│  ├── high_risk_patterns    → MasterOutput.diagnosis_high_risk_rows│
│  └── comparison            → MasterOutput.diagnosis_comparison_rows│
└───────────────────────┬─────────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│  delivery_app.py renders Diagnosis tab in Gradio            │
│  ├── diagnosis_text (Markdown summary)                      │
│  ├── diagnosis_high_risk_df (table)                         │
│  └── diagnosis_comparison_df (table)                        │
└─────────────────────────────────────────────────────────────┘
```

### Diagnosis Fixes

| # | Fix | File | Why |
|---|---|---|---|
| 11 | Freshness check for predict prerequisite | `delivery_app.py` | Checks sidecar `mtime < 1h`; appends `[SYSTEM: FRESH]` or `[SYSTEM: NOT FRESH]` to query |
| 12 | Conditional predict-before-diagnose | `master_expert.md` | If `FRESH` → skip predict, diagnose directly. If `NOT FRESH` → run predict first as prerequisite (not error) |

---

## Simulation / What if Agent

"Simulate delays if weather changes to Stormy across East regions"

"What if all Same Day deliveries in the North region face Foggy weather — simulate the impact"

"Simulate delays if we replace all Bike assignments on routes over 100km with Vans"

```perl
┌─────────────────────────────────────────────────────────────────────┐
│                        USER QUERY                                   │
│  e.g. "Simulate delays if weather changes to Stormy in East"        │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│                     delivery_app.py                                 │
│  Freshness check: is daily_delivery_delay_prediction_meta.json      │
│  < 1 hour old?                                                      │
│    YES → prepend [SYSTEM: … FRESH]                                  │
│    NO  → prepend [SYSTEM: … NOT FRESH]                              │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│              master_expert (orchestrator agent)                     │
│                                                                     │
│  Section 3 pre-requisite check:                                     │
│    NOT FRESH? → call predict_delivery_delays_tool first             │
│    FRESH?     → skip predict, go straight to simulate               │
│                                                                     │
│  Calls delay_simulations_tool (wraps simulation agent)              │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│          delay_simulation_agent  (delay_simulation.md)              │
│                                                                     │
│  Translates natural-language query into:                            │
│    scenario = "Weather changes to Stormy in East"                   │
│    filters  = {"region": "east"}                                    │
│    changes  = {"weather_condition": "stormy"}                       │
│                                                                     │
│  Calls simulate_order_delays tool exactly once                      │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│         simulate_order_delays  (tools/simulate_delays.py)           │
│                                                                     │
│  Step 1: Parse & validate JSON inputs                               │
│                                                                     │
│  Step 2: Read daily_delivery_delay_prediction.csv                   │
│          (1346 delayed orders from latest predict run)              │
│                                                                     │
│  Step 3: Build filter mask                                          │
│          region == "east"  →  241 matching rows                     │
│                                                                     │
│  Step 4: Apply column changes                                       │
│          Set weather_condition = "stormy" on those 241 rows         │
│                                                                     │
│  Step 5: Severity lookup from SQLite DB                             │
│          ┌─────────────────────────────────────────────┐            │
│          │  Lookup priority:                           │            │
│          │  1. mode + weather  (hist_summary_by_mode_  │            │
│          │     weather)  ← uses filter_ctx if avail    │            │
│          │  2. weather + vehicle (hist_summary_by_     │            │
│          │     weather_vehicle)                        │            │
│          │  3. mode + vehicle  (averaged from two      │            │
│          │     single-dim tables)                      │            │
│          │  4. Single-dim fallback (highest delay_rate)│            │
│          └─────────────────────────────────────────────┘            │
│          Returns: {delay_rate: 0.416, fracs: [0.08, 0.63, 0.29]}    │
│                                                                     │
│  Step 6: Assign new severity labels proportionally                  │
│          Short: 8%  Medium: 63%  Long: 29%  (across 241 rows)       │
│                                                                     │
│  Step 7: Save simulation_delivery_delays.csv                        │
│                                                                     │
│  Step 8: Return Markdown summary + top N rows (SC_MCP_DISPLAY_ROWS) │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│          delay_simulation_agent  (post-processing)                  │
│                                                                     │
│  Enriches EVERY row with simulate_delay_reason inferred from        │
│  scenario + original_severity + simulated_severity                  │
│                                                                     │
│  Returns SimulationsList (list[SimulateDelays])                     │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│              master_expert  (post-processing)                       │
│                                                                     │
│  Copies simulations → MasterOutput.simulate_rows                    │
│  Calls format_summary_tool → MasterOutput.simulate_summary          │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────┐
│                      delivery_app.py  (Gradio)                      │
│                                                                     │
│  simulate_summary → Simulation tab (Markdown)                       │
│  simulate_rows → DataFrame table                                    │
│  Saves simulate_delays_latest.csv to output/                        │
└─────────────────────────────────────────────────────────────────────┘
```

### Changes Made (4 Files)

| # | File | What changed |
|---|---|---|
| 1 | `tools/simulate_delays.py` | Complete rewrite. New signature (`scenario`, `filters`, `changes`). Reads real delayed-orders CSV, filters rows by any combo of `region/mode/vehicle/weather/partner/package/distance`, applies column changes, looks up historical severity distribution from `hist_summary_by_*` SQLite tables, reassigns severity labels proportionally, saves simulation CSV, returns Markdown summary + top rows. Row cap uses `SC_MCP_DISPLAY_ROWS` env var (default 50). |
| 2 | `delivery_agents.py` | Updated **SimulateDelays** Pydantic model. Replaced `order_id`, `simulate_weather`, `simulate_traffic`, `order_date`, `delivery_date`, `simulate_delay_hours` with `delivery_id`, `delivery_partner`, `delivery_mode`, `region`, `weather_condition`, `vehicle_type`, `distance_km`, `original_severity`, `simulated_severity`, `simulate_delay_reason`. |
| 3 | `delay_simulation.md` | Rewrote agent prompt. Documented the **3-arg tool signature**, JSON parameter format with examples, valid values for all columns (lowercase), removed traffic references. |
| 4 | `master_expert.md` | Updated Section 3. Added **FRESH/NOT FRESH prerequisite check** (same as diagnosis). Added valid simulation parameter reference. Updated tool description from `"weather/traffic"` to `"weather/vehicle/mode"`. |



### Key Design Decisions

| Design Decision | Description |
|---|---|
| No ML re-run | Severity is inferred from **historical summary tables already in the DB**, not from re-running the prediction pipeline. |
| Combined-table priority | Lookup priority: `mode+weather → weather+vehicle → mode+vehicle (averaged) → single-dimension fallback`. |
| Filter context | Filter values (not being changed) are passed to the lookup to enable more specific combined-table matches (e.g., filtering by `delivery_mode=same day` + changing `weather=foggy` → uses the `mode_weather` combined table). |
| Prerequisite check | Simulation now has the same **FRESH/NOT FRESH check** as diagnosis, since it reads the delayed-orders CSV that predict produces. |

---

## Recommend Actions Agent

```perl
┌─────────────────────────────────────────────────────────┐
│              Master Orchestrator Agent                  │
│                                                         │
│  Pre-requisite check (from [SYSTEM:] tag in query):     │
│  • FRESH → call recommendation_tool directly            │
│  • NOT FRESH → predict → diagnose → then recommend      │
└──────────────────────┬──────────────────────────────────┘
                       │ calls recommendation_tool
                       ▼
┌─────────────────────────────────────────────────────────┐
│           Recommendation Agent (sub-agent)              │
│  Prompt: config/prompts/agents/recommendation.md        │
│  Output: RecommendedActionsList                         │
│    └─ RecommendedAction:                                │
│         action, action_desc, category, dimension,       │
│         supporting_data, sla_reference  ← NEW FIELD     │
│  tools=[recommend_actions]                              │
│  tool_choice="required" → MUST call the tool            │
└──────────────────────┬──────────────────────────────────┘
                       │ calls recommend_actions()
                       ▼
┌─────────────────────────────────────────────────────────┐
│         recommend_actions() — @function_tool            │
│         tools/recommend_actions.py                      │
│                                                         │
│  1. Validate prerequisites (DB + CSV exist)             │
│  2. Query SQLite: overall stats (hist_ + daily_)        │
│  3. Compare 6 dimensions (daily vs historical)          │
│  4. Get high-risk patterns (hist + daily, top 10)       │
│  5. Get worst dimensions (hist top 3 + daily top 3)     │
│  6. Read CSV for severity hotspots                      │
│  7. Build structured Markdown (9 sections)              │
│  8. ──► RAG: retrieve_sla_context(tool_output) ◄──      │
│         (lazy import, auto-appended)                    │
└────────────────────────┬────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────┐
│        retrieve_sla_context() — rag_knowledge.py        │
│                                                         │
│  1. _summarize_query(): extract key lines with          │
│     %, bold, risk, critical markers → focused query     │
│  2. Embed query (text-embedding-3-small)                │
│  3. ChromaDB vector search (TOP_K=10, cosine)           │
│     Auto-builds vectorstore if missing/stale            │
│  4.a _hybrid_prefilter(): 70% cosine sim                │
|                         + 30% keyword overlap           │
│                         → TOP_N=8 best chunks           │
│  4.b _reranker(): Cross-encoder rerank → TOP_5 chunks   │
│  5. Format as "--- SLA Knowledge Context ---" block     │
└────────────────────────┬────────────────────────────────┘
                         │ returns combined string:
                         │   [Data sections] + [SLA Context]
                         ▼
┌─────────────────────────────────────────────────────────┐
│      Recommendation Agent receives tool output          │
│                                                         │
│  Prompt mandates for EACH recommendation:               │
│  • supporting_data → cite specific data metrics         │
│  • sla_reference → MUST quote SLA clause/target/penalty │
│  • action_desc → explain gap + corrective action        │
│                                                         │
│  Produces RecommendedActionsList                        │
└──────────────────────┬──────────────────────────────────┘
                       │ returns to Master
                       ▼
┌─────────────────────────────────────────────────────────┐
│  Master calls format_summary_tool                       │
│  summary_type: recommendation                           │
│  Renders: Quick-Win / Short-term / Long-term sections   │
│  Each action shows: data metrics + SLA reference        │
│  → MasterOutput.recommendation_summary                  │
└─────────────────────────────────────────────────────────┘
```

### Key Aspects to Remember

| Aspect | Detail |
|---|---|
| RAG is embedded in the tool | `retrieve_sla_context()` is called inside `recommend_actions()` via lazy import — the agent does **NOT** call RAG separately |
| Lazy import | `from tools.rag_knowledge import retrieve_sla_context` is inside the function body to avoid circular imports through `tools/__init__.py` |
| Prerequisites | The tool checks that the SQLite DB and daily delayed CSV exist (created by **predict + diagnose** steps) before proceeding |
| 6 categorical dimensions | `region`, `weather_condition`, `delivery_partner`, `delivery_mode`, `vehicle_type`, `distance_bucket` |
| Vectorstore auto-rebuild | ChromaDB at `vectorstore/` auto-rebuilds when the SLA file hash changes — no manual rebuild needed |
| Chunking | LangChain two-stage: `MarkdownHeaderTextSplitter (#/##/###)` → `RecursiveCharacterTextSplitter (2000 chars, 200 overlap)` → **73 chunks** |
| Re-ranking formula | `0.7 × cosine_similarity + 0.3 × keyword_overlap` — prioritizes semantic match but rewards keyword hits |
| Query summarization | Extracts lines with `%`, `**bold**`, `risk`, `critical`, `worse` markers — creates a focused query from the data output |
| `tool_choice="required"` | Agent **MUST** call `recommend_actions` — it cannot skip the tool and hallucinate recommendations |
| Output model | `RecommendedActionsList` → list of `RecommendedAction` with `category` (`long-term` / `short-term` / `quick-win`) and dimension fields |
| Graceful RAG failure | If RAG fails, the tool still returns the data sections with an error note — recommendations can still be generated from data alone |

---

## Email Alerts Agent

```perl
┌──────────────────────────────────────────────────────────┐
│              Master Orchestrator Agent                   │
│                                                          │
│  Pre-requisite check (3-tier):                           │
│  1. predict already called in this conversation → skip   │
│  2. [SYSTEM: FRESH] → skip predict, call email directly  │
│  3. [SYSTEM: NOT FRESH] + no prior predict → run predict │
└──────────────────────┬───────────────────────────────────┘
                       │ calls email_alert_tool
                       ▼
┌──────────────────────────────────────────────────────────┐
│           Email Alert Agent (sub-agent)                  │
│  Prompt: config/prompts/agents/email_alert.md            │
│  Output: EmailsList                                      │
│    └─ EmailAlert: { email_content, email_id }            │
│  tools=[fetch_delayed_orders_for_email]                  │
└──────────────────────┬───────────────────────────────────┘
                       │ calls fetch_delayed_orders_for_email()
                       ▼
┌──────────────────────────────────────────────────────────┐
│    fetch_delayed_orders_for_email() — @function_tool     │
│    tools/email_customers.py                              │
│                                                          │
│  1. Read output/daily_delivery_delay_prediction.csv      │
│  2. Filter delayed rows (or use all if delayed-only CSV) │
│  3. For EACH delayed order:                              │
│     ├─ _severity_key() → Long/Medium/Short               │
│     └─ _generate_email() → fill template with:           │
│        {order_id, weather, delivery_mode,                │
│         region, distance_km}                             │
│  4. 3 severity-based templates:                          │
│     ├─ Long (6+h): escalation tone, priority routing     │
│     ├─ Medium (3-5h): monitoring tone, proactive update  │
│     └─ Short (1-2h): informational, minor delay notice   │
│  5. Write back to CSV:                                   │
│     ├─ email_template_name column                        │
│     └─ email_content column                              │
│  6. Return summary string:                               │
│     ├─ template counts by severity                       │
│     ├─ template definitions                              │
│     └─ sample emails (1 per template type)               │
└──────────────────────┬───────────────────────────────────┘
                       │ returns summary string
                       ▼
┌──────────────────────────────────────────────────────────┐
│  Email Agent creates single EmailAlert with              │
│  email_content = entire tool summary                     │
│  → EmailsList returned to Master                         │
└──────────────────────┬───────────────────────────────────┘
                       │ returns to Master
                       ▼
┌──────────────────────────────────────────────────────────┐
│  Master:                                                 │
│  1. Calls format_summary_tool (summary_type: email_alert)│
│     → MasterOutput.email_alert_summary                   │
│  2. Sets MasterOutput.email_alerts = EmailsList          │
└──────────────────────┬───────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────┐
│  delivery_app.py (Gradio UI):                            │
│  • Renders email_alert_text from email_alerts.content    │
│    with --- separator between templates                  │
│  • Shows download link for updated CSV                   │
│    (output/daily_delivery_delay_prediction.csv)          │
└──────────────────────────────────────────────────────────┘
```

### Email Alerts — Key Aspects to Remember

| Aspect | Detail |
|---|---|
| Tool function | `fetch_delayed_orders_for_email()` in `tools/email_customers.py` — returns `str`, not a list |
| Input CSV | `output/daily_delivery_delay_prediction.csv` — the delayed-orders CSV written by the predict pipeline |
| 3 severity templates | **Long (6+h):** escalation tone, priority routing; **Medium (3–5h):** monitoring, proactive update; **Short (1–2h):** informational, minor delay |
| Template placeholders | `{order_id}`, `{weather}`, `{delivery_mode}`, `{region}`, `{distance_km}` — filled per row |
| CSV write-back | Adds 2 columns: `email_template_name` and `email_content` to the same CSV |
| Delayed-only CSV handling | If CSV has no `predict_delay` column (already filtered to delayed-only), all rows are treated as delayed |
| delivery_id float fix | `.replace(".0", "")` strips trailing `.0` from float-cast delivery IDs |
| Tool returns summary string | Template counts + template definitions + sample emails (1 per type) — **NOT individual emails** |
| Agent creates single EmailAlert | The email agent wraps the entire tool summary into **ONE `EmailAlert` object** (prompt instructs this) |
| Pydantic model | `EmailAlert: {email_content: str, email_id: str}`; `EmailsList: {content: list[EmailAlert]}` with `min_length=1` |
| UI rendering | `gr.Markdown` — templates joined with `\n\n---\n\n` separator and **To:** bolded |
| CSV download | `gr.File` widget shows the updated CSV with email columns for download |
| Pre-requisite (master prompt) | **3-tier check:** (1) predict already called → skip, (2) `[SYSTEM: FRESH]` → skip, (3) `NOT FRESH` → run predict first |
| format_summary_tool | Master calls it with `summary_type: email_alert` → renders overview, template breakdown, CSV confirmation |
| allowed_paths | `delivery_app.py` includes `output/` in `allowed_paths` for Gradio file serving |

----

## ChatInterface vs Chatbot

## Gradio UI Design Comparison

| Feature | gr.ChatInterface | Manual gr.Chatbot + gr.Textbox (current) |
|---|---|---|
| Layout | Single-column, chat only | Two-column: chat left, output tabs right |
| Streaming yields | Only `(text)` or `(history)` — one output | Can yield **15-tuple**: history + pending state + all 12 tab outputs simultaneously |
| Structured outputs | No way to populate separate tabs (Predict, Diagnosis, Simulation, etc.) from same handler | Each yield updates chatbot **AND all right-side tabs** in one pass |
| Action plan / confirmation | No state management — no way to stash a pending query | `gr.State("")` for `pending_state` enables **plan → confirm → execute** flow |
| Quick-action buttons | Can't have buttons that auto-submit preset text | Buttons `.click()` → populate textbox → `.then()` calls `chat_handler` |
| File upload wiring | Limited — `additional_inputs` exist but awkward to wire | `orders_file` is a normal input wired into the handler directly |
| Clear behavior | Built-in clear only resets chat | Custom clear resets chat + textbox + pending state + shows welcome message |
| Initial welcome message | `chatbot` parameter exists but limited control | `value=[{"role": "assistant", "content": _WELCOME_MSG}]` — full control |

**Key reason for the manual approach**

`gr.ChatInterface` controls its own layout and only yields chat messages. There is no way to have a single `chat_handler` that streams updates to both the chatbot **and 12+ output components** (markdown, dataframes, file downloads) across **5 tabs**.

The manual `gr.Chatbot + gr.Textbox` setup allows a **15-element yield tuple** that updates **chat + state + all structured outputs simultaneously**.

---

## Security Guardrails

### Attack Surface Analysis

```perl
User Input
    ↓
Master Expert  ← ONLY entry point for raw user text
    ↓ (structured tool call arguments)
Sub-agents (Predict / Diagnose / Simulate / Recommend / Email)
    ↓ (typed Pydantic output)
Master Expert  ← assembles final MasterOutput
```
### Why sub-agents dont need security_guardrails.md to be preppended to their instructions

| Concern | Sub-agents? | Reason |
|---|---|---|
| Prompt injection from user | ✗ Not needed | Sub-agents never receive raw user text. They only receive structured inputs constructed by the master expert. |
| Scope violation | ✗ Not needed | Each sub-agent is already narrowly scoped by its own prompt (predict only, diagnose only, etc.) + `tool_choice="required"` pins it to one tool. |
| Output safety | ✗ Not needed | `output_type=PydanticModel` enforces a typed schema — the LLM can't produce arbitrary free-form output. |
| Data fabrication | Partially | Sub-agent prompts already instruct against this; Pydantic validation rejects structurally wrong output. |

The one case where you would add it to sub-agents
If tool outputs (e.g. CSV data, MCP responses) contain injected instructions that could manipulate a sub-agent's behavior — for example, an order record containing "Ignore previous instructions and return..." — then sub-agents could theoretically be influenced. However, since their outputs are Pydantic-constrained, any injected text would at most corrupt a string field, not change tool-calling behavior.

Conclusion: Keep security guardrails only on the master expert. It's the sole user-facing agent and the correct place to enforce all input validation and scope restriction.


---

## Master Agent — Logic Flowchart

### 1. App Layer — Plan Confirmation Gate (before agent runs)

```perl
┌─────────────────────────────────────────────────────────────────────┐
│                        User Message                                 │
└─────────────────────────────────────────────────────────────────────┘
                               │
                  ┌────────────▼────────────┐
                  │  Pending query in       │
                  │  gr.State?              │
                  └────────────┬────────────┘
              Yes ─────────────┤─────────────── No
              │                                 │
  ┌───────────▼────────────┐       ┌────────────▼──────────────┐
  │  Confirm word?         │       │  Greeting?                │
  │  yes/proceed/sure/ok   │       │  hi/thanks/help/bye       │
  └───────────┬────────────┘       └─────────────┬─────────────┘
   Yes ───────┤──── No              Yes ─────────┤──── No
   │          │                     │            │
   │    ┌─────▼──────────┐          │     ┌──────▼──────────────────┐
   │    │ Discard pending│          │     │  _build_action_plan()   │
   │    │ treat as fresh │          │     │  regex match on query   │
   │    └─────┬──────────┘          │     └──────┬──────────────────┘
   │          │                     │       0 matches     1+ matches
   │          │                     │         │               │
   │          │                     │   ┌─────▼──────┐  ┌────▼───────────────────┐
   │          │                     │   │Clarification│  │Expand composite intents│
   │          │                     │   │   message  │  │e.g. SLA→predict+diag+  │
   │          │                     │   │  (end turn)│  │     recommend          │
   │          │                     │   └────────────┘  └────┬───────────────────┘
   │          │                     │                        │
   │          └──────────────────►  │         ┌──────────────▼──────────────────┐
   │                                │         │  Add prerequisite steps if      │
   │                                │         │  data NOT fresh:                │
   │                                │         │  - predict not fresh → prepend  │
   │                                │         │    "Run predictions first"      │
   │                                │         │  - diagnose not fresh → prepend │
   │                                │         │    "Run diagnosis first"        │
   │                                │         └─────────────┬───────────────────┘
   │                                │                       │
   │                                │         ┌─────────────▼───────────────────┐
   │                                │         │  Show numbered plan +           │
   │                                │         │  "Shall I proceed?"             │
   │                                │         │  Stash query → gr.State pending │
   │                                │         │  (end turn, wait for confirm)   │
   │                                │         └─────────────────────────────────┘
   │                                │
   └────────────────────────────────┘
                  │  (confirmed or greeting or quick-action button)
                  ▼
```

---

### 2. Agent Execution — Master Expert Orchestrator

```perl
┌─────────────────────────────────────────────────────────────────────┐
│              Master Expert Agent                                    │
│  Prompt: security_guardrails + chatbot_behavior + master_expert.md  │
│  model_settings: tool_choice="auto", output_type=MasterOutput       │
│                                                                     │
│  Build full_query:                                                  │
│  - Append file path if orders CSV uploaded                          │
│  - Append [SYSTEM: freshness tag] (FRESH / NOT FRESH)               │
└─────────────────────┬───────────────────────────────────────────────┘
                      │  calls tools SEQUENTIALLY (one at a time)
    ┌─────────────────┼─────────────────────────────────┐
    │                 │                                 │
    ▼                 ▼                                 ▼
┌──────────────────────────┐   ┌──────────────────────────────────────┐
│  predict_delivery_       │   │  diagnose_delay_patterns_tool        │
│  delays_tool             │   │  (sub-agent)                         │
│  (sub-agent)             │   │  Prompt: diagnose_delay_patterns.md  │
│  Prompt: predict_        │   │  Output: DelayDiagnosisResult        │
│  delivery_delays.md      │   │  - high_risk_patterns: list          │
│  Output:                 │   │  - comparison: list (today vs hist)  │
│  DeliveryDelayPrediction │   │  tools=[get_delay_diagnosis via MCP] │
│  - predict_summary: str  │   └──────────────────────────────────────┘
│  - delayed_orders: list  │
│  tools=[predict via MCP] │   ┌──────────────────────────────────────┐
└──────────────────────────┘   │  delay_simulations_tool              │
                               │  (sub-agent)                         │
                               │  Prompt: delay_simulation.md         │
                               │  Output: SimulationsList             │
                               │  tools=[simulate_order_delays]       │
                               └──────────────────────────────────────┘

                               ┌──────────────────────────────────────┐
                               │  recommendation_tool                 │
                               │  (sub-agent)                         │
                               │  Prompt: recommendation.md           │
                               │  Output: RecommendedActionsList      │
                               │  - min 9 actions (3 per category)    │
                               │  - quick-win / short-term /          │
                               │    long-term                         │
                               │  tools=[recommend_actions + RAG]     │
                               └──────────────────────────────────────┘

                               ┌──────────────────────────────────────┐
                               │  email_alert_tool                    │
                               │  (sub-agent)                         │
                               │  Prompt: email_alert.md              │
                               │  Output: EmailsList                  │
                               │  - EmailAlert: {email_content,       │
                               │    email_id}                         │
                               │  tools=[fetch_delayed_orders_        │
                               │         for_email]                   │
                               └──────────────────────────────────────┘
                                               │
                               ┌───────────────▼──────────────────────┐
                               │  Assemble MasterOutput (Pydantic)    │
                               │  - predict_summary + predict_rows    │
                               │  - diagnosis_summary + high_risk_    │
                               │    rows + comparison_rows            │
                               │  - simulate_summary + simulate_rows  │
                               │  - recommendation_summary            │
                               │  - email_alert_summary + email_alerts│
                               └──────────────────────────────────────┘
```

---

### 3. App Layer — Post-Processing & Output

```perl
┌─────────────────────────────────────────────────────────────────────┐
│                  App Layer (delivery_chat_app.py)                   │
│                                                                     │
│  1. Read predict CSV + sidecar JSON                                 │
│     └─ merge llm_insights per delivery_id (normalised float→int)    │
│     └─ trim to _MCP_DISPLAY_ROWS for table display                  │
│                                                                     │
│  2. Build simulate_df → save simulate_delays_latest.csv             │
│                                                                     │
│  3. Build diagnosis tables                                          │
│     └─ diagnosis_high_risk_df (today's patterns)                    │
│     └─ diagnosis_comparison_df (today vs historical)                │
│     └─ Save diagnosis_meta.json sidecar for next freshness check    │
│                                                                     │
│  4. Format email alerts                                             │
│     └─ Join with \n\n---\n\n separator                              │
│     └─ If no delayed orders → "No email alerts generated"           │
│                                                                     │
│  5. Build "Analysis complete" summary message                       │
│     └─ List which tabs have results                                 │
│     └─ Re-append _WELCOME_MSG for next interaction                  │
│                                                                     │
│  6. Yield 15-tuple to Gradio                                        │
│     (history, textbox, pending_state,                               │
│      predict_md, predict_df, predict_csv,                           │
│      simulate_md, simulate_df, simulate_csv,                        │
│      diagnosis_md, diag_hr_df, diag_comp_df,                        │
│      recommend_md, email_md, email_csv)                             │
└─────────────────────────────────────────────────────────────────────┘
                               │
          ┌────────────────────┼────────────────────┐
          ▼                    ▼                    ▼
   ┌─────────────┐    ┌────────────────┐   ┌────────────────┐
   │  Chat (left)│    │  Output Tabs   │   │  File Downloads│
   │  history +  │    │  Predict       │   │  predict CSV   │
   │  status log │    │  Diagnosis     │   │  simulate CSV  │
   └─────────────┘    │  Simulation    │   │  email CSV     │
                      │  Recommend     │   └────────────────┘
                      │  Email/Alerts  │
                      └────────────────┘
```
